# SqueezeNet Comprehensive Evaluation

This notebook provides a comprehensive evaluation of SqueezeNet for edge AI deployment, including:
- Performance benchmarking
- Latency analysis
- Memory profiling
- Batch processing evaluation
- Comparison with MobileNetV2

## Model Overview
- **Architecture**: SqueezeNet 1.1
- **Parameters**: ~1.2M
- **Size**: ~4.8 MB
- **Key Features**: Fire modules, reduced parameters, competitive accuracy

In [ ]:
# Install required packages
!pip install torch torchvision psutil pandas matplotlib seaborn numpy pillow tqdm requests

import torch
import torchvision.models as models
import torchvision.transforms as transforms
import time
import psutil
import os
import sys
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from tqdm import tqdm
import json
from pathlib import Path
import requests
from PIL import Image

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✓ All packages imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {torch.cuda.is_available() or torch.backends.mps.is_available()}")

## 1. Model Setup and Device Configuration

In [ ]:
# Device configuration
if torch.backends.mps.is_available():
    device = torch.device("mps")
    device_name = "Apple Silicon (MPS)"
elif torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = f"CUDA GPU ({torch.cuda.get_device_name()})"
else:
    device = torch.device("cpu")
    device_name = "CPU"

print(f"Using device: {device_name}")

# Load SqueezeNet model
print("Loading SqueezeNet 1.1...")
model = models.squeezenet1_1(pretrained=True)
model.eval()
model.to(device)

# Calculate model size
def calculate_model_size(model):
    param_size = 0
    buffer_size = 0
    
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    
    model_size_mb = (param_size + buffer_size) / (1024 * 1024)
    return model_size_mb

model_size = calculate_model_size(model)
print(f"✓ Model loaded successfully")
print(f"Model size: {model_size:.2f} MB")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Prepare Test Data and Preprocessing

In [ ]:
# Image preprocessing pipeline
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create synthetic test data
test_input = torch.randn(1, 3, 224, 224).to(device)

# Test with a real image if available
try:
    # Try to load a test image
    test_image_path = "../jupyter/test_images/test_dog.jpeg"
    if os.path.exists(test_image_path):
        image = Image.open(test_image_path).convert('RGB')
        real_input = transform(image).unsqueeze(0).to(device)
        print(f"✓ Loaded real test image: {test_image_path}")
    else:
        real_input = test_input
        print("Using synthetic test data")
except Exception as e:
    real_input = test_input
    print(f"Using synthetic test data (real image load failed: {e})")

print(f"Test input shape: {test_input.shape}")
print(f"Memory allocated on device: {torch.cuda.memory_allocated() / 1024**2:.1f} MB" if device.type == 'cuda' else "N/A (CPU/MPS)")

## 3. Core Performance Benchmark

In [ ]:
def benchmark_inference(model, test_input, device, num_runs=100, warmup_runs=10):
    """Comprehensive inference benchmarking"""
    print(f"Running benchmark: {warmup_runs} warmup + {num_runs} measurement runs")
    
    # Warmup
    with torch.no_grad():
        for _ in tqdm(range(warmup_runs), desc="Warmup"):
            _ = model(test_input)
    
    # Synchronize if using GPU
    if device.type in ['cuda', 'mps']:
        if device.type == 'cuda':
            torch.cuda.synchronize()
    
    # Benchmark
    inference_times = []
    cpu_usage = []
    memory_usage = []
    gpu_memory_usage = []
    process = psutil.Process()
    
    for i in tqdm(range(num_runs), desc="Benchmarking"):
        # System metrics before
        cpu_before = process.cpu_percent()
        mem_before = process.memory_info().rss / (1024 * 1024)
        
        if device.type == 'cuda':
            gpu_mem_before = torch.cuda.memory_allocated() / (1024 * 1024)
        else:
            gpu_mem_before = 0
        
        # Inference timing
        start_time = time.perf_counter()
        
        with torch.no_grad():
            output = model(test_input)
        
        # Synchronize if using GPU
        if device.type == 'cuda':
            torch.cuda.synchronize()
        
        end_time = time.perf_counter()
        
        # System metrics after
        cpu_after = process.cpu_percent()
        mem_after = process.memory_info().rss / (1024 * 1024)
        
        if device.type == 'cuda':
            gpu_mem_after = torch.cuda.memory_allocated() / (1024 * 1024)
        else:
            gpu_mem_after = 0
        
        # Record measurements
        inference_times.append((end_time - start_time) * 1000)  # Convert to ms
        cpu_usage.append(cpu_after)
        memory_usage.append(mem_after)
        gpu_memory_usage.append(gpu_mem_after)
    
    return {
        'times': inference_times,
        'cpu_usage': cpu_usage,
        'memory_usage': memory_usage,
        'gpu_memory': gpu_memory_usage,
        'output_shape': output.shape
    }

# Run core benchmark
benchmark_results = benchmark_inference(model, test_input, device, num_runs=100)

# Calculate statistics
times = benchmark_results['times']
stats = {
    'mean': np.mean(times),
    'std': np.std(times),
    'min': np.min(times),
    'max': np.max(times),
    'p50': np.percentile(times, 50),
    'p95': np.percentile(times, 95),
    'p99': np.percentile(times, 99)
}

print("\n=== CORE PERFORMANCE RESULTS ===")
print(f"Mean inference time: {stats['mean']:.2f} ± {stats['std']:.2f} ms")
print(f"Min/Max: {stats['min']:.2f} / {stats['max']:.2f} ms")
print(f"Percentiles: P50={stats['p50']:.2f}, P95={stats['p95']:.2f}, P99={stats['p99']:.2f} ms")
print(f"Throughput: {1000/stats['mean']:.1f} images/second")
print(f"Memory usage: {np.mean(benchmark_results['memory_usage']):.1f} MB")
print(f"CPU usage: {np.mean(benchmark_results['cpu_usage']):.1f}%")
if device.type == 'cuda':
    print(f"GPU memory: {np.mean(benchmark_results['gpu_memory']):.1f} MB")

## 4. Visualization: Performance Distribution

In [ ]:
# Create comprehensive performance visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('SqueezeNet Performance Analysis', fontsize=16)

# 1. Inference time distribution
axes[0, 0].hist(benchmark_results['times'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].axvline(stats['mean'], color='red', linestyle='--', label=f'Mean: {stats["mean"]:.2f}ms')
axes[0, 0].axvline(stats['p95'], color='orange', linestyle='--', label=f'P95: {stats["p95"]:.2f}ms')
axes[0, 0].set_xlabel('Inference Time (ms)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Inference Time Distribution')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Time series plot
run_numbers = range(1, len(benchmark_results['times']) + 1)
axes[0, 1].plot(run_numbers, benchmark_results['times'], alpha=0.7, linewidth=1)
axes[0, 1].axhline(stats['mean'], color='red', linestyle='--', label=f'Mean: {stats["mean"]:.2f}ms')
axes[0, 1].fill_between(run_numbers, 
                       stats['mean'] - stats['std'], 
                       stats['mean'] + stats['std'], 
                       alpha=0.2, color='red', label='±1 std')
axes[0, 1].set_xlabel('Run Number')
axes[0, 1].set_ylabel('Inference Time (ms)')
axes[0, 1].set_title('Inference Time Over Runs')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Memory usage
axes[1, 0].plot(run_numbers, benchmark_results['memory_usage'], color='green', alpha=0.7)
axes[1, 0].set_xlabel('Run Number')
axes[1, 0].set_ylabel('Memory Usage (MB)')
axes[1, 0].set_title('Memory Usage Over Time')
axes[1, 0].grid(True, alpha=0.3)

# 4. CPU usage
axes[1, 1].plot(run_numbers, benchmark_results['cpu_usage'], color='purple', alpha=0.7)
axes[1, 1].set_xlabel('Run Number')
axes[1, 1].set_ylabel('CPU Usage (%)')
axes[1, 1].set_title('CPU Usage Over Time')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics table
summary_df = pd.DataFrame({
    'Metric': ['Mean Time (ms)', 'Std Dev (ms)', 'Min Time (ms)', 'Max Time (ms)', 
               'P95 Time (ms)', 'P99 Time (ms)', 'Throughput (img/s)', 'Memory (MB)', 'CPU (%)'],
    'Value': [stats['mean'], stats['std'], stats['min'], stats['max'], 
              stats['p95'], stats['p99'], 1000/stats['mean'], 
              np.mean(benchmark_results['memory_usage']), np.mean(benchmark_results['cpu_usage'])]
})

summary_df['Value'] = summary_df['Value'].round(2)
print("\n=== PERFORMANCE SUMMARY ===")
print(summary_df.to_string(index=False))

## 5. Batch Processing Analysis

In [ ]:
def benchmark_batch_sizes(model, device, batch_sizes=[1, 5, 10, 20, 50]):
    """Test different batch sizes"""
    batch_results = {}
    
    for batch_size in batch_sizes:
        print(f"Testing batch size: {batch_size}")
        
        # Create batch input
        batch_input = torch.randn(batch_size, 3, 224, 224).to(device)
        
        # Warmup
        with torch.no_grad():
            for _ in range(5):
                _ = model(batch_input)
        
        # Benchmark
        times = []
        memory_usage = []
        
        for _ in range(10):
            start_time = time.perf_counter()
            
            with torch.no_grad():
                output = model(batch_input)
            
            if device.type == 'cuda':
                torch.cuda.synchronize()
            
            end_time = time.perf_counter()
            
            batch_time = (end_time - start_time) * 1000
            times.append(batch_time)
            
            if device.type == 'cuda':
                memory_usage.append(torch.cuda.memory_allocated() / (1024 * 1024))
            else:
                memory_usage.append(psutil.Process().memory_info().rss / (1024 * 1024))
        
        avg_time = np.mean(times)
        time_per_image = avg_time / batch_size
        throughput = batch_size / (avg_time / 1000)
        
        batch_results[batch_size] = {
            'total_time': avg_time,
            'time_per_image': time_per_image,
            'throughput': throughput,
            'memory_usage': np.mean(memory_usage),
            'efficiency': (1 / time_per_image) * 1000  # Images per second per image
        }
    
    return batch_results

# Run batch size analysis
batch_results = benchmark_batch_sizes(model, device)

# Create batch analysis DataFrame
batch_df = pd.DataFrame(batch_results).T
batch_df.index.name = 'Batch Size'

print("\n=== BATCH PROCESSING RESULTS ===")
print(batch_df.round(2))

# Visualize batch performance
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Batch Processing Performance', fontsize=16)

batch_sizes = list(batch_results.keys())

# Throughput vs batch size
throughputs = [batch_results[bs]['throughput'] for bs in batch_sizes]
axes[0, 0].plot(batch_sizes, throughputs, 'o-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Batch Size')
axes[0, 0].set_ylabel('Throughput (images/sec)')
axes[0, 0].set_title('Throughput vs Batch Size')
axes[0, 0].grid(True, alpha=0.3)

# Time per image vs batch size
times_per_image = [batch_results[bs]['time_per_image'] for bs in batch_sizes]
axes[0, 1].plot(batch_sizes, times_per_image, 'o-', color='red', linewidth=2, markersize=8)
axes[0, 1].set_xlabel('Batch Size')
axes[0, 1].set_ylabel('Time per Image (ms)')
axes[0, 1].set_title('Latency vs Batch Size')
axes[0, 1].grid(True, alpha=0.3)

# Memory usage vs batch size
memory_usages = [batch_results[bs]['memory_usage'] for bs in batch_sizes]
axes[1, 0].plot(batch_sizes, memory_usages, 'o-', color='green', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('Batch Size')
axes[1, 0].set_ylabel('Memory Usage (MB)')
axes[1, 0].set_title('Memory Usage vs Batch Size')
axes[1, 0].grid(True, alpha=0.3)

# Efficiency comparison
total_times = [batch_results[bs]['total_time'] for bs in batch_sizes]
axes[1, 1].plot(batch_sizes, total_times, 'o-', color='purple', linewidth=2, markersize=8)
axes[1, 1].set_xlabel('Batch Size')
axes[1, 1].set_ylabel('Total Time (ms)')
axes[1, 1].set_title('Total Processing Time vs Batch Size')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Network Latency Tolerance Analysis

In [ ]:
def benchmark_network_latency(model, test_input, device, network_delays=[0, 50, 100, 200, 500, 1000]):
    """Test performance with simulated network delays"""
    latency_results = {}
    
    for delay in network_delays:
        print(f"Testing with {delay}ms network delay...")
        
        times = []
        inference_times = []
        
        for _ in range(20):  # Smaller sample for latency tests
            # Simulate pre-processing network delay
            start_total = time.perf_counter()
            time.sleep(delay / 1000.0)
            
            # Actual inference
            start_inference = time.perf_counter()
            with torch.no_grad():
                output = model(test_input)
            
            if device.type == 'cuda':
                torch.cuda.synchronize()
            
            end_inference = time.perf_counter()
            
            # Simulate post-processing network delay
            time.sleep(delay / 1000.0)
            end_total = time.perf_counter()
            
            inference_time = (end_inference - start_inference) * 1000
            total_time = (end_total - start_total) * 1000
            
            times.append(total_time)
            inference_times.append(inference_time)
        
        avg_inference = np.mean(inference_times)
        avg_total = np.mean(times)
        efficiency_ratio = (avg_inference / avg_total) * 100
        
        latency_results[delay] = {
            'inference_time': avg_inference,
            'total_time': avg_total,
            'network_overhead': avg_total - avg_inference,
            'efficiency_ratio': efficiency_ratio
        }
    
    return latency_results

# Run network latency analysis
latency_results = benchmark_network_latency(model, test_input, device)

# Create latency analysis DataFrame
latency_df = pd.DataFrame(latency_results).T
latency_df.index.name = 'Network Delay (ms)'

print("\n=== NETWORK LATENCY ANALYSIS ===")
print(latency_df.round(2))

# Visualize latency tolerance
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Network Latency Tolerance Analysis', fontsize=16)

delays = list(latency_results.keys())

# Total time vs network delay
total_times = [latency_results[d]['total_time'] for d in delays]
inference_times = [latency_results[d]['inference_time'] for d in delays]

axes[0, 0].plot(delays, total_times, 'o-', label='Total Time', linewidth=2, markersize=8)
axes[0, 0].plot(delays, inference_times, 'o-', label='Inference Time', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Network Delay (ms)')
axes[0, 0].set_ylabel('Time (ms)')
axes[0, 0].set_title('Processing Time vs Network Delay')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Efficiency ratio
efficiency_ratios = [latency_results[d]['efficiency_ratio'] for d in delays]
axes[0, 1].plot(delays, efficiency_ratios, 'o-', color='red', linewidth=2, markersize=8)
axes[0, 1].set_xlabel('Network Delay (ms)')
axes[0, 1].set_ylabel('Efficiency (%)')
axes[0, 1].set_title('Computational Efficiency vs Network Delay')
axes[0, 1].grid(True, alpha=0.3)

# Network overhead
network_overheads = [latency_results[d]['network_overhead'] for d in delays]
axes[1, 0].plot(delays, network_overheads, 'o-', color='green', linewidth=2, markersize=8)
axes[1, 0].set_xlabel('Network Delay (ms)')
axes[1, 0].set_ylabel('Network Overhead (ms)')
axes[1, 0].set_title('Network Overhead vs Delay')
axes[1, 0].grid(True, alpha=0.3)

# Stacked bar chart
axes[1, 1].bar(delays, inference_times, label='Inference Time', alpha=0.8)
axes[1, 1].bar(delays, network_overheads, bottom=inference_times, label='Network Overhead', alpha=0.8)
axes[1, 1].set_xlabel('Network Delay (ms)')
axes[1, 1].set_ylabel('Time (ms)')
axes[1, 1].set_title('Time Composition by Network Delay')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Model Comparison with MobileNetV2

In [ ]:
# Load existing MobileNetV2 results if available
def load_mobilenet_results():
    """Load the latest MobileNetV2 results for comparison"""
    results_dir = Path("results")
    if results_dir.exists():
        mobilenet_files = list(results_dir.glob("mobilenetv2_summary_*.csv"))
        if mobilenet_files:
            latest_file = max(mobilenet_files, key=lambda x: x.stat().st_mtime)
            return pd.read_csv(latest_file)
    return None

# Quick MobileNetV2 benchmark for comparison
def quick_mobilenet_benchmark():
    """Run a quick MobileNetV2 benchmark for comparison"""
    try:
        import tensorflow as tf
        
        # Check if model exists
        model_path = "mobilenetv2.tflite"
        if not os.path.exists(model_path):
            model_path = "../mobilenetv2.tflite"
        
        if os.path.exists(model_path):
            print("Running quick MobileNetV2 benchmark for comparison...")
            
            # Load TFLite model
            interpreter = tf.lite.Interpreter(model_path=model_path)
            interpreter.allocate_tensors()
            
            input_details = interpreter.get_input_details()
            output_details = interpreter.get_output_details()
            
            # Prepare input
            test_input_tflite = np.random.randint(0, 255, (1, 224, 224, 3), dtype=np.uint8)
            
            # Warmup
            for _ in range(10):
                interpreter.set_tensor(input_details[0]['index'], test_input_tflite)
                interpreter.invoke()
            
            # Benchmark
            times = []
            for _ in range(50):
                start = time.perf_counter()
                interpreter.set_tensor(input_details[0]['index'], test_input_tflite)
                interpreter.invoke()
                end = time.perf_counter()
                times.append((end - start) * 1000)
            
            return {
                'mean_time': np.mean(times),
                'model_size': os.path.getsize(model_path) / (1024 * 1024),
                'device': 'CPU (TFLite)'
            }
        else:
            print("MobileNetV2 model not found for comparison")
            return None
    except Exception as e:
        print(f"Could not benchmark MobileNetV2: {e}")
        return None

# Try to get MobileNetV2 comparison data
mobilenet_data = load_mobilenet_results()
if mobilenet_data is None:
    mobilenet_quick = quick_mobilenet_benchmark()
else:
    mobilenet_quick = {
        'mean_time': mobilenet_data['Mean_Inference_Time_ms'].iloc[0],
        'model_size': mobilenet_data['Model_Size_MB'].iloc[0],
        'device': 'CPU (TFLite)'
    }

# Create comparison DataFrame
comparison_data = {
    'Model': ['SqueezeNet', 'MobileNetV2'],
    'Size (MB)': [model_size, mobilenet_quick['model_size'] if mobilenet_quick else 'N/A'],
    'Inference Time (ms)': [stats['mean'], mobilenet_quick['mean_time'] if mobilenet_quick else 'N/A'],
    'Throughput (img/s)': [1000/stats['mean'], 1000/mobilenet_quick['mean_time'] if mobilenet_quick else 'N/A'],
    'Device': [device_name, mobilenet_quick['device'] if mobilenet_quick else 'N/A'],
    'Framework': ['PyTorch', 'TensorFlow Lite']
}

comparison_df = pd.DataFrame(comparison_data)

print("\n=== MODEL COMPARISON ===")
print(comparison_df.to_string(index=False))

# Visualization if we have MobileNetV2 data
if mobilenet_quick:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('SqueezeNet vs MobileNetV2 Comparison', fontsize=16)
    
    models = ['SqueezeNet', 'MobileNetV2']
    sizes = [model_size, mobilenet_quick['model_size']]
    times = [stats['mean'], mobilenet_quick['mean_time']]
    throughputs = [1000/stats['mean'], 1000/mobilenet_quick['mean_time']]
    
    # Model size comparison
    bars1 = axes[0].bar(models, sizes, color=['skyblue', 'lightcoral'], alpha=0.8)
    axes[0].set_ylabel('Model Size (MB)')
    axes[0].set_title('Model Size Comparison')
    axes[0].grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, size in zip(bars1, sizes):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    f'{size:.1f}', ha='center', va='bottom')
    
    # Inference time comparison
    bars2 = axes[1].bar(models, times, color=['skyblue', 'lightcoral'], alpha=0.8)
    axes[1].set_ylabel('Inference Time (ms)')
    axes[1].set_title('Inference Speed Comparison')
    axes[1].grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, time_val in zip(bars2, times):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    f'{time_val:.1f}', ha='center', va='bottom')
    
    # Throughput comparison
    bars3 = axes[2].bar(models, throughputs, color=['skyblue', 'lightcoral'], alpha=0.8)
    axes[2].set_ylabel('Throughput (images/sec)')
    axes[2].set_title('Throughput Comparison')
    axes[2].grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, throughput in zip(bars3, throughputs):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{throughput:.1f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    # Performance ratio analysis
    size_ratio = model_size / mobilenet_quick['model_size']
    speed_ratio = mobilenet_quick['mean_time'] / stats['mean']  # Higher is better for SqueezeNet
    
    print(f"\n=== PERFORMANCE RATIOS ===")
    print(f"SqueezeNet is {size_ratio:.2f}x {'larger' if size_ratio > 1 else 'smaller'} than MobileNetV2")
    print(f"SqueezeNet is {speed_ratio:.2f}x {'faster' if speed_ratio > 1 else 'slower'} than MobileNetV2")
    print(f"Speed/Size efficiency: {speed_ratio/size_ratio:.2f}")

## 8. Quantization Analysis (if supported)

In [ ]:
def benchmark_quantization():
    """Test quantized vs non-quantized model performance"""
    print("Testing quantization effects...")
    
    # Original model results (already have these)
    original_time = stats['mean']
    original_size = model_size
    
    try:
        # Create quantized model
        print("Creating quantized model...")
        model_quantized = models.squeezenet1_1(pretrained=True)
        model_quantized.eval()
        
        # Apply dynamic quantization
        model_quantized = torch.quantization.quantize_dynamic(
            model_quantized, {torch.nn.Linear, torch.nn.Conv2d}, dtype=torch.qint8
        )
        
        # Note: For quantized models, we typically use CPU
        device_quant = torch.device("cpu")
        model_quantized.to(device_quant)
        test_input_cpu = torch.randn(1, 3, 224, 224).to(device_quant)
        
        # Calculate quantized model size
        quantized_size = calculate_model_size(model_quantized)
        
        # Benchmark quantized model
        quant_times = []
        
        # Warmup
        with torch.no_grad():
            for _ in range(10):
                _ = model_quantized(test_input_cpu)
        
        # Benchmark
        for _ in range(50):
            start = time.perf_counter()
            with torch.no_grad():
                _ = model_quantized(test_input_cpu)
            quant_times.append((time.perf_counter() - start) * 1000)
        
        quantized_time = np.mean(quant_times)
        
        # Results
        quantization_results = {
            'original': {
                'size_mb': original_size,
                'inference_time_ms': original_time,
                'device': str(device)
            },
            'quantized': {
                'size_mb': quantized_size,
                'inference_time_ms': quantized_time,
                'device': 'CPU'
            },
            'ratios': {
                'size_reduction': original_size / quantized_size,
                'speed_change': quantized_time / original_time  # Note: might be slower due to CPU vs GPU
            }
        }
        
        return quantization_results
        
    except Exception as e:
        print(f"Quantization analysis failed: {e}")
        return None

# Run quantization analysis
quant_results = benchmark_quantization()

if quant_results:
    print("\n=== QUANTIZATION ANALYSIS ===")
    print(f"Original model (FP32): {quant_results['original']['size_mb']:.2f} MB, "
          f"{quant_results['original']['inference_time_ms']:.2f} ms ({quant_results['original']['device']})")
    print(f"Quantized model (INT8): {quant_results['quantized']['size_mb']:.2f} MB, "
          f"{quant_results['quantized']['inference_time_ms']:.2f} ms ({quant_results['quantized']['device']})")
    print(f"Size reduction: {quant_results['ratios']['size_reduction']:.2f}x")
    print(f"Speed change: {quant_results['ratios']['speed_change']:.2f}x")
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('Quantization Effects', fontsize=16)
    
    models = ['FP32 Original', 'INT8 Quantized']
    sizes = [quant_results['original']['size_mb'], quant_results['quantized']['size_mb']]
    times = [quant_results['original']['inference_time_ms'], quant_results['quantized']['inference_time_ms']]
    
    # Size comparison
    bars1 = axes[0].bar(models, sizes, color=['lightblue', 'orange'], alpha=0.8)
    axes[0].set_ylabel('Model Size (MB)')
    axes[0].set_title('Model Size: FP32 vs INT8')
    axes[0].grid(True, alpha=0.3)
    
    for bar, size in zip(bars1, sizes):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    f'{size:.1f}', ha='center', va='bottom')
    
    # Time comparison
    bars2 = axes[1].bar(models, times, color=['lightblue', 'orange'], alpha=0.8)
    axes[1].set_ylabel('Inference Time (ms)')
    axes[1].set_title('Inference Time: FP32 vs INT8')
    axes[1].grid(True, alpha=0.3)
    
    for bar, time_val in zip(bars2, times):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    f'{time_val:.1f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

## 9. Save Comprehensive Results

In [ ]:
# Compile all results in standardized format
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Comprehensive results for JSON export
comprehensive_results = {
    'timestamp': datetime.now().isoformat(),
    'device_info': {
        'system': 'Darwin',  # You can use platform.system() for cross-platform
        'release': '24.5.0',  # platform.release()
        'machine': 'x86_64',  # platform.machine()
        'processor': device_name,
        'cpu_cores': psutil.cpu_count(),
        'total_ram_gb': psutil.virtual_memory().total / (1024**3),
        'python_version': f"Python {'.'.join(str(x) for x in sys.version_info[:3])}",
        'pytorch_version': torch.__version__,
        'device_type': str(device).replace(':', '')
    },
    'model_info': {
        'name': 'SqueezeNet',
        'type': 'Lightweight CNN',
        'size_mb': model_size,
        'input_shape': [1, 3, 224, 224],
        'output_shape': list(benchmark_results['output_shape']),
        'quantized': False,  # Update this if you implement quantization
        'device': str(device)
    },
    'core_performance': {
        'inference_times': benchmark_results['times'],
        'statistics': {
            'mean_inference_time': stats['mean'],
            'std_inference_time': stats['std'],
            'min_inference_time': stats['min'],
            'max_inference_time': stats['max'],
            'p95_inference_time': stats['p95'],
            'p99_inference_time': stats['p99'],
            'mean_memory_usage': np.mean(benchmark_results['memory_usage']),
            'peak_memory_usage': np.max(benchmark_results['memory_usage']),
            'mean_cpu_usage': np.mean(benchmark_results['cpu_usage']),
            'peak_cpu_usage': np.max(benchmark_results['cpu_usage'])
        },
        'memory_usage': benchmark_results['memory_usage'],
        'cpu_usage': benchmark_results['cpu_usage']
    },
    'batch_processing': batch_results,
    'latency_analysis': latency_results,
    'quantization': quant_results if quant_results else {},
    'comparison': {
        'mobilenetv2': mobilenet_quick if mobilenet_quick else {}
    }
}

# Calculate cold start overhead (using first vs average of later runs)
cold_start_overhead = benchmark_results['times'][0] - stats['mean']

# Create results directory
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

# Save comprehensive JSON
json_file = results_dir / f"squeezenet_results_{timestamp}.json"
with open(json_file, 'w') as f:
    json.dump(comprehensive_results, f, indent=2)

# Save summary CSV in standardized format
summary_data = {
    'Model': ['SqueezeNet'],
    'Device_Tier': ['Laptop (Tier 2)'],
    'Device_Type': [str(device).replace(':', '')],
    'Model_Size_MB': [model_size],
    'Quantized': [False],  # Update if quantization is implemented
    'Mean_Inference_Time_ms': [stats['mean']],
    'Std_Inference_Time_ms': [stats['std']],
    'P95_Inference_Time_ms': [stats['p95']],
    'Mean_Memory_Usage_MB': [np.mean(benchmark_results['memory_usage'])],
    'Peak_Memory_Usage_MB': [np.max(benchmark_results['memory_usage'])],
    'Mean_CPU_Usage_Percent': [np.mean(benchmark_results['cpu_usage'])],
    'GPU_Memory_MB': [np.mean(benchmark_results['gpu_memory']) if device.type == 'cuda' else 0],
    'Cold_Start_Overhead_ms': [cold_start_overhead],
    'Max_Throughput_imgs_per_sec': [max([batch_results[bs]['throughput'] for bs in batch_results.keys()])],
    'Efficiency_No_Network_Percent': [latency_results[0]['efficiency_ratio']],
    'Efficiency_500ms_Network_Percent': [latency_results[500]['efficiency_ratio']]
}

summary_df = pd.DataFrame(summary_data)
csv_file = results_dir / f"squeezenet_summary_{timestamp}.csv"
summary_df.to_csv(csv_file, index=False)

# Save detailed benchmark data for analysis
benchmark_df = pd.DataFrame({
    'Run': range(1, len(benchmark_results['times']) + 1),
    'Inference_Time_ms': benchmark_results['times'],
    'Memory_Usage_MB': benchmark_results['memory_usage'],
    'CPU_Usage_Percent': benchmark_results['cpu_usage'],
    'GPU_Memory_MB': benchmark_results['gpu_memory'] if device.type == 'cuda' else [0] * len(benchmark_results['times'])
})
detailed_csv = results_dir / f"squeezenet_detailed_{timestamp}.csv"
benchmark_df.to_csv(detailed_csv, index=False)

print(f"\n=== RESULTS SAVED ===")
print(f"Comprehensive JSON: {json_file}")
print(f"Summary CSV: {csv_file}")
print(f"Detailed CSV: {detailed_csv}")

# Display final summary in standardized format
print(f"\n=== SQUEEZENET EVALUATION SUMMARY ===")
print(f"Device: {device_name}")
print(f"Model Size: {model_size:.2f} MB")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Quantized: False")
print()
print("PERFORMANCE METRICS:")
print(f"  Mean Inference Time: {stats['mean']:.2f} ± {stats['std']:.2f} ms")
print(f"  95th Percentile: {stats['p95']:.2f} ms")
print(f"  Memory Usage: {np.mean(benchmark_results['memory_usage']):.1f} MB (peak: {np.max(benchmark_results['memory_usage']):.1f} MB)")
print(f"  CPU Usage: {np.mean(benchmark_results['cpu_usage']):.1f}% (peak: {np.max(benchmark_results['cpu_usage']):.1f}%)")
if device.type == 'cuda':
    print(f"  GPU Memory: {np.mean(benchmark_results['gpu_memory']):.1f} MB")
print()
print("DELAY-TOLERANT ANALYSIS:")
print(f"  Cold Start Overhead: {cold_start_overhead:.2f} ms")
print(f"  Max Throughput: {max([batch_results[bs]['throughput'] for bs in batch_results.keys()]):.1f} images/sec")
print(f"  Efficiency (no network): {latency_results[0]['efficiency_ratio']:.1f}%")
print(f"  Efficiency (500ms network): {latency_results[500]['efficiency_ratio']:.1f}%")

if mobilenet_quick:
    speed_ratio = mobilenet_quick['mean_time'] / stats['mean']
    size_ratio = model_size / mobilenet_quick['model_size']
    print(f"\nCOMPARISON TO MOBILENETV2:")
    print(f"  Speed: {speed_ratio:.2f}x {'faster' if speed_ratio > 1 else 'slower'}")
    print(f"  Size: {size_ratio:.2f}x {'larger' if size_ratio > 1 else 'smaller'}")
    print(f"  Speed/Size efficiency: {speed_ratio/size_ratio:.2f}")

if quant_results:
    print(f"\nQUANTIZATION EFFECTS:")
    print(f"  Size reduction: {quant_results['ratios']['size_reduction']:.2f}x")
    print(f"  Speed change: {quant_results['ratios']['speed_change']:.2f}x")

print(f"\n✓ SqueezeNet comprehensive evaluation completed!")
print(f"✓ Results saved in standardized format for cross-model comparison")